In [7]:
!nvidia-smi

Tue Sep 15 09:41:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [9]:
!mkdir -p gpu_tictactoe
!cd gpu_tictactoe && pwd

/content/gpu_tictactoe


In [10]:
!ls -la gpu_tictactoe

total 28
drwxr-xr-x 2 root root  4096 Sep 15 09:40 .
drwxr-xr-x 1 root root  4096 Sep 15 09:40 ..
-rw-r--r-- 1 root root 17877 Sep 15 09:40 gpu_tictactoe.cu


In [11]:
%%writefile gpu_tictactoe/gpu_tictactoe.cu

#include <cuda_runtime.h>

#include <iostream>
#include <iomanip>
#include <thread>
#include <mutex>
#include <condition_variable>
#include <string>
#include <cstdlib>

#define BOARD_SIZE 9

#define EMPTY 0
#define PLAYER_X 1
#define PLAYER_O 2

// ============================================================
// CUDA ERROR CHECKING
// ============================================================

void checkCudaError(cudaError_t error, const char* message)
{
    if (error != cudaSuccess)
    {
        std::cerr << "\nCUDA Error: "
                  << message
                  << " - "
                  << cudaGetErrorString(error)
                  << std::endl;

        std::exit(EXIT_FAILURE);
    }
}

// ============================================================
// SHARED GAME STATE
// ============================================================

int board[BOARD_SIZE] =
{
    EMPTY, EMPTY, EMPTY,
    EMPTY, EMPTY, EMPTY,
    EMPTY, EMPTY, EMPTY
};

int currentPlayer = PLAYER_X;
int moveNumber = 1;

bool gameFinished = false;

// Mutex protects the shared board and turn information.
std::mutex gameMutex;

// Condition variable controls which player can move.
std::condition_variable turnCondition;

// ============================================================
// DEVICE FUNCTION
// ============================================================

// Checks whether placing a player's symbol at a position
// creates a winning Tic-Tac-Toe line.
__device__ bool isWinningMove(
    const int* board,
    int position,
    int player)
{
    int tempBoard[BOARD_SIZE];

    for (int i = 0; i < BOARD_SIZE; i++)
    {
        tempBoard[i] = board[i];
    }

    tempBoard[position] = player;

    int winningLines[8][3] =
    {
        {0, 1, 2},
        {3, 4, 5},
        {6, 7, 8},
        {0, 3, 6},
        {1, 4, 7},
        {2, 5, 8},
        {0, 4, 7},
        {2, 4, 6}
    };

    for (int i = 0; i < 8; i++)
    {
        int a = winningLines[i][0];
        int b = winningLines[i][1];
        int c = winningLines[i][2];

        if (tempBoard[a] == player &&
            tempBoard[b] == player &&
            tempBoard[c] == player)
        {
            return true;
        }
    }

    return false;
}

// ============================================================
// PLAYER X CUDA KERNEL
// ============================================================

// Every CUDA thread evaluates one board position.
//
// X strategy:
// 100 = immediate winning move
// 50  = center
// 30  = corner
// 10  = other available position
// -1  = occupied position

__global__ void evaluatePlayerX(
    const int* board,
    int* scores)
{
    int position =
        blockIdx.x * blockDim.x + threadIdx.x;

    if (position >= BOARD_SIZE)
        return;

    if (board[position] != EMPTY)
    {
        scores[position] = -1;
        return;
    }

    if (isWinningMove(
            board,
            position,
            PLAYER_X))
    {
        scores[position] = 100;
        return;
    }

    if (position == 4)
    {
        scores[position] = 50;
        return;
    }

    if (position == 0 ||
        position == 2 ||
        position == 6 ||
        position == 8)
    {
        scores[position] = 30;
        return;
    }

    scores[position] = 10;
}

// ============================================================
// PLAYER O CUDA KERNEL
// ============================================================

// O strategy:
// 100 = immediate winning move
// 90  = block X
// 50  = center
// 30  = corner
// 10  = other available position
// -1  = occupied position

__global__ void evaluatePlayerO(
    const int* board,
    int* scores)
{
    int position =
        blockIdx.x * blockDim.x + threadIdx.x;

    if (position >= BOARD_SIZE)
        return;

    if (board[position] != EMPTY)
    {
        scores[position] = -1;
        return;
    }

    // First priority: O wins.
    if (isWinningMove(
            board,
            position,
            PLAYER_O))
    {
        scores[position] = 100;
        return;
    }

    // Second priority: block X.
    if (isWinningMove(
            board,
            position,
            PLAYER_X))
    {
        scores[position] = 90;
        return;
    }

    if (position == 4)
    {
        scores[position] = 50;
        return;
    }

    if (position == 0 ||
        position == 2 ||
        position == 6 ||
        position == 8)
    {
        scores[position] = 30;
        return;
    }

    scores[position] = 10;
}

// ============================================================
// DISPLAY BOARD
// ============================================================

void displayBoard(const int* board)
{
    std::cout << "\n";

    std::cout << "       1   2   3\n";
    std::cout << "     +---+---+---+\n";

    for (int row = 0; row < 3; row++)
    {
        std::cout << "  "
                  << row + 1
                  << "  |";

        for (int col = 0; col < 3; col++)
        {
            int position =
                row * 3 + col;

            char symbol = ' ';

            if (board[position] == PLAYER_X)
                symbol = 'X';

            else if (board[position] == PLAYER_O)
                symbol = 'O';

            std::cout << " "
                      << symbol
                      << " |";
        }

        std::cout << "\n";
        std::cout << "     +---+---+---+\n";
    }

    std::cout << "\n";
}

// ============================================================
// CHECK WINNER
// ============================================================

int checkWinner(const int* board)
{
    int winningLines[8][3] =
    {
        {0, 1, 2},
        {3, 4, 5},
        {6, 7, 8},
        {0, 3, 6},
        {1, 4, 7},
        {2, 5, 8},
        {0, 4, 7},
        {2, 4, 6}
    };

    for (int i = 0; i < 8; i++)
    {
        int a = winningLines[i][0];
        int b = winningLines[i][1];
        int c = winningLines[i][2];

        if (board[a] != EMPTY &&
            board[a] == board[b] &&
            board[b] == board[c])
        {
            return board[a];
        }
    }

    return EMPTY;
}

// ============================================================
// CHECK DRAW
// ============================================================

bool isBoardFull(const int* board)
{
    for (int i = 0; i < BOARD_SIZE; i++)
    {
        if (board[i] == EMPTY)
            return false;
    }

    return true;
}

// ============================================================
// SELECT BEST MOVE
// ============================================================

int selectBestMove(const int* scores)
{
    int bestPosition = -1;
    int bestScore = -1;

    for (int i = 0; i < BOARD_SIZE; i++)
    {
        if (scores[i] > bestScore)
        {
            bestScore = scores[i];
            bestPosition = i;
        }
    }

    return bestPosition;
}

// ============================================================
// GPU PLAYER X THREAD
// ============================================================

void playerXThread()
{
    // Allocate GPU memory for this competitor.
    int* deviceBoard = nullptr;
    int* deviceScores = nullptr;

    checkCudaError(
        cudaMalloc(
            (void**)&deviceBoard,
            BOARD_SIZE * sizeof(int)),
        "Allocating Player X board memory");

    checkCudaError(
        cudaMalloc(
            (void**)&deviceScores,
            BOARD_SIZE * sizeof(int)),
        "Allocating Player X score memory");

    int scores[BOARD_SIZE];

    while (true)
    {
        std::unique_lock<std::mutex> lock(gameMutex);

        turnCondition.wait(
            lock,
            []()
            {
                return currentPlayer == PLAYER_X ||
                       gameFinished;
            });

        if (gameFinished)
        {
            break;
        }

        std::cout << "\n";
        std::cout << "==================================================\n";
        std::cout << "TURN "
                  << moveNumber
                  << ": GPU PLAYER X THREAD\n";
        std::cout << "==================================================\n";

        std::cout << "GPU Competitor: X\n";
        std::cout << "Strategy: Offensive\n";
        std::cout << "Synchronization: X has acquired the game lock.\n";
        std::cout << "CUDA kernel: 9 threads evaluating 9 positions.\n";

        // Copy shared board to GPU.
        checkCudaError(
            cudaMemcpy(
                deviceBoard,
                board,
                BOARD_SIZE * sizeof(int),
                cudaMemcpyHostToDevice),
            "Copying board for Player X");

        // Launch CUDA kernel.
        evaluatePlayerX<<<1, 9>>>(
            deviceBoard,
            deviceScores);

        checkCudaError(
            cudaGetLastError(),
            "Launching Player X kernel");

        checkCudaError(
            cudaDeviceSynchronize(),
            "Synchronizing Player X kernel");

        // Copy GPU scores back to CPU.
        checkCudaError(
            cudaMemcpy(
                scores,
                deviceScores,
                BOARD_SIZE * sizeof(int),
                cudaMemcpyDeviceToHost),
            "Copying Player X scores");

        int selectedMove =
            selectBestMove(scores);

        if (selectedMove == -1)
        {
            gameFinished = true;
            turnCondition.notify_all();
            break;
        }

        board[selectedMove] = PLAYER_X;

        std::cout << "GPU Player X selected position "
                  << selectedMove + 1
                  << ".\n";

        std::cout << "Move score: "
                  << scores[selectedMove]
                  << "\n";

        displayBoard(board);

        int winner = checkWinner(board);

        if (winner != EMPTY)
        {
            std::cout << "==================================================\n";
            std::cout << "WINNER: GPU PLAYER X\n";
            std::cout << "==================================================\n";

            gameFinished = true;
            turnCondition.notify_all();
            break;
        }

        if (isBoardFull(board))
        {
            std::cout << "==================================================\n";
            std::cout << "RESULT: DRAW\n";
            std::cout << "==================================================\n";

            gameFinished = true;
            turnCondition.notify_all();
            break;
        }

        // Pass control to Player O.
        currentPlayer = PLAYER_O;
        moveNumber++;

        std::cout << "Synchronization: X releases the game lock.\n";

        lock.unlock();

        turnCondition.notify_all();
    }

    cudaFree(deviceBoard);
    cudaFree(deviceScores);
}

// ============================================================
// GPU PLAYER O THREAD
// ============================================================

void playerOThread()
{
    // Allocate GPU memory for this competitor.
    int* deviceBoard = nullptr;
    int* deviceScores = nullptr;

    checkCudaError(
        cudaMalloc(
            (void**)&deviceBoard,
            BOARD_SIZE * sizeof(int)),
        "Allocating Player O board memory");

    checkCudaError(
        cudaMalloc(
            (void**)&deviceScores,
            BOARD_SIZE * sizeof(int)),
        "Allocating Player O score memory");

    int scores[BOARD_SIZE];

    while (true)
    {
        std::unique_lock<std::mutex> lock(gameMutex);

        turnCondition.wait(
            lock,
            []()
            {
                return currentPlayer == PLAYER_O ||
                       gameFinished;
            });

        if (gameFinished)
        {
            break;
        }

        std::cout << "\n";
        std::cout << "==================================================\n";
        std::cout << "TURN "
                  << moveNumber
                  << ": GPU PLAYER O THREAD\n";
        std::cout << "==================================================\n";

        std::cout << "GPU Competitor: O\n";
        std::cout << "Strategy: Defensive\n";
        std::cout << "Synchronization: O has acquired the game lock.\n";
        std::cout << "CUDA kernel: 9 threads evaluating 9 positions.\n";

        // Copy shared board to GPU.
        checkCudaError(
            cudaMemcpy(
                deviceBoard,
                board,
                BOARD_SIZE * sizeof(int),
                cudaMemcpyHostToDevice),
            "Copying board for Player O");

        // Launch CUDA kernel.
        evaluatePlayerO<<<1, 9>>>(
            deviceBoard,
            deviceScores);

        checkCudaError(
            cudaGetLastError(),
            "Launching Player O kernel");

        checkCudaError(
            cudaDeviceSynchronize(),
            "Synchronizing Player O kernel");

        // Copy scores back to CPU.
        checkCudaError(
            cudaMemcpy(
                scores,
                deviceScores,
                BOARD_SIZE * sizeof(int),
                cudaMemcpyDeviceToHost),
            "Copying Player O scores");

        int selectedMove =
            selectBestMove(scores);

        if (selectedMove == -1)
        {
            gameFinished = true;
            turnCondition.notify_all();
            break;
        }

        board[selectedMove] = PLAYER_O;

        std::cout << "GPU Player O selected position "
                  << selectedMove + 1
                  << ".\n";

        std::cout << "Move score: "
                  << scores[selectedMove]
                  << "\n";

        displayBoard(board);

        int winner = checkWinner(board);

        if (winner != EMPTY)
        {
            std::cout << "==================================================\n";
            std::cout << "WINNER: GPU PLAYER O\n";
            std::cout << "==================================================\n";

            gameFinished = true;
            turnCondition.notify_all();
            break;
        }

        if (isBoardFull(board))
        {
            std::cout << "==================================================\n";
            std::cout << "RESULT: DRAW\n";
            std::cout << "==================================================\n";

            gameFinished = true;
            turnCondition.notify_all();
            break;
        }

        // Pass control back to Player X.
        currentPlayer = PLAYER_X;
        moveNumber++;

        std::cout << "Synchronization: O releases the game lock.\n";

        lock.unlock();

        turnCondition.notify_all();
    }

    cudaFree(deviceBoard);
    cudaFree(deviceScores);
}

// ============================================================
// MAIN
// ============================================================

int main()
{
    std::cout << "==================================================\n";
    std::cout << "       TWO GPU COMPETITOR TIC-TAC-TOE\n";
    std::cout << "==================================================\n";

    // --------------------------------------------------------
    // Detect GPU.
    // --------------------------------------------------------

    int deviceCount = 0;

    checkCudaError(
        cudaGetDeviceCount(&deviceCount),
        "Getting CUDA device count");

    std::cout << "\nCUDA devices available: "
              << deviceCount
              << "\n";

    if (deviceCount == 0)
    {
        std::cerr << "No CUDA GPU detected.\n";
        return 1;
    }

    cudaDeviceProp deviceProperties;

    checkCudaError(
        cudaGetDeviceProperties(
            &deviceProperties,
            0),
        "Getting GPU properties");

    std::cout << "GPU Device: "
              << deviceProperties.name
              << "\n";

    std::cout << "CUDA Capability: "
              << deviceProperties.major
              << "."
              << deviceProperties.minor
              << "\n";

    // --------------------------------------------------------
    // Explain architecture.
    // --------------------------------------------------------

    std::cout << "\nGPU COMPETITOR ARCHITECTURE\n";
    std::cout << "----------------------------------------------\n";

    std::cout << "Logical GPU Competitor X -> Offensive Strategy\n";
    std::cout << "Logical GPU Competitor O -> Defensive Strategy\n";

    std::cout << "\nThis environment provides one physical GPU.\n";
    std::cout << "The two GPU competitors run as separate host\n";
    std::cout << "threads and take synchronized turns using CUDA.\n";

    std::cout << "\nStarting synchronized GPU game...\n";

    displayBoard(board);

    // --------------------------------------------------------
    // Create two competitor threads.
    // --------------------------------------------------------

    std::thread playerX(playerXThread);
    std::thread playerO(playerOThread);

    // --------------------------------------------------------
    // Wait for both competitors.
    // --------------------------------------------------------

    playerX.join();
    playerO.join();

    // --------------------------------------------------------
    // Final board.
    // --------------------------------------------------------

    std::cout << "\n==================================================\n";
    std::cout << "FINAL GAME BOARD\n";
    std::cout << "==================================================\n";

    displayBoard(board);

    std::cout << "\nBoth GPU competitor threads completed.\n";
    std::cout << "Shared board synchronization completed.\n";
    std::cout << "CUDA memory released successfully.\n";

    std::cout << "\n==================================================\n";
    std::cout << "          GAME EXECUTION COMPLETED\n";
    std::cout << "==================================================\n";

    return 0;
}

Overwriting gpu_tictactoe/gpu_tictactoe.cu


In [12]:
!nvcc -std=c++17 gpu_tictactoe/gpu_tictactoe.cu -o gpu_tictactoe_app

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [13]:
!./gpu_tictactoe_app

       TWO GPU COMPETITOR TIC-TAC-TOE

CUDA devices available: 1
GPU Device: Tesla T4
CUDA Capability: 7.5

GPU COMPETITOR ARCHITECTURE
----------------------------------------------
Logical GPU Competitor X -> Offensive Strategy
Logical GPU Competitor O -> Defensive Strategy

This environment provides one physical GPU.
The two GPU competitors run as separate host
threads and take synchronized turns using CUDA.

Starting synchronized GPU game...

       1   2   3
     +---+---+---+
  1  |   |   |   |
     +---+---+---+
  2  |   |   |   |
     +---+---+---+
  3  |   |   |   |
     +---+---+---+


TURN 1: GPU PLAYER X THREAD
GPU Competitor: X
Strategy: Offensive
Synchronization: X has acquired the game lock.
CUDA kernel: 9 threads evaluating 9 positions.
GPU Player X selected position 5.
Move score: 50

       1   2   3
     +---+---+---+
  1  |   |   |   |
     +---+---+---+
  2  |   | X |   |
     +---+---+---+
  3  |   |   |   |
     +---+---+---+

Synchronization: X releases the game